# Analysis of Regional Deployments

This notebook analyzes CSV files across different regions and deployments, examining:
- Number of files per deployment
- Date ranges for each deployment
- Distribution of records by date

In [ ]:
# Import required libraries
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import glob
import os
from matplotlib.backends.backend_pdf import PdfPages

# Set plotting style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)


## Summary

The notebook above will:

1. Find all regions matching the pattern `/gws/ssde/j25b/ceh_generic/kgoldmann/{region}_inferences/`
2. For each region, find all deployment directories (matching `dep*`)
3. For each deployment:
   - Count and print the number of CSV files
   - Read all CSV files and extract the date column
   - Collect all dates into a DataFrame
   - Print the start and end date for that deployment
   - Plot a bar chart showing the distribution of records by date
   - Display the top 5 dates by record count

All results are stored in the `results` dictionary for further analysis if needed.

In [ ]:
base_path = Path('/gws/ssde/j25b/ceh_generic/kgoldmann/')

regions = os.listdir(base_path)
regions = [r.replace('_inferences', '') for r in regions if r.endswith('_inferences')]

regions

In [ ]:
def analyze_deployment(deployment_path, date_col = 'image_datetime' ):
    """
    Analyze a single deployment directory.
    Returns dict with file count, dates DataFrame, start date, end date.
    """
    # Find all CSV files in the deployment directory
    csv_files = list(deployment_path.glob("*.csv"))
    file_count = len(csv_files)

    if file_count == 0:
        return None

    # Collect all dates from all CSV files
    all_dates = []

    for csv_file in csv_files:
        try:
            # Read CSV and extract date column
            df = pd.read_csv(csv_file)

            if date_col:
                # Parse dates and add to collection
                dates = pd.to_datetime(df[date_col], errors='coerce')
                all_dates.extend(dates.dropna().tolist())
        except Exception as e:
            print(f"    Warning: Could not read {csv_file.name}: {e}")
            continue

    if not all_dates:
        return {'file_count': file_count, 'dates': None, 'start_date': None, 'end_date': None}

    # Create DataFrame of dates
    dates_df = pd.DataFrame({'date': all_dates})
    dates_df['date'] = pd.to_datetime(dates_df['date'])

    # Get date range
    start_date = dates_df['date'].min()
    end_date = dates_df['date'].max()

    return {
        'file_count': file_count,
        'dates': dates_df,
        'start_date': start_date,
        'end_date': end_date
    }

In [ ]:
# Main analysis loop
results = {}
summary_data = []  # List to collect summary data for table

# Create PDF for all plots
pdf_filename = 'deployment_plots.pdf'
pdf = PdfPages(pdf_filename)

for region in sorted(regions):
    region_path = base_path / f"{region}_inferences"

    print(f"\n{'='*80}")
    print(f"REGION: {region}")
    print(f"{'='*80}")

    # Find all deployment directories
    deployments = sorted([d for d in region_path.glob('dep*') if d.is_dir()])

    if not deployments:
        print(f"No deployment directories found!")
        continue

    results[region] = {}

    for deployment in deployments:
        print(f"\n  Deployment: {deployment.name}")
        print(f"  {'-'*60}")

        analysis = analyze_deployment(deployment)

        if analysis is None:
            print(f"No CSV files found!")
            continue

        results[region][deployment.name] = analysis

        # Collect summary data for table
        summary_row = {
            'Region': region,
            'Deployment': deployment.name,
            'File Count': analysis['file_count'],
            'Start Date': analysis['start_date'].strftime('%Y-%m-%d') if analysis['start_date'] else 'N/A',
            'End Date': analysis['end_date'].strftime('%Y-%m-%d') if analysis['end_date'] else 'N/A',
            'Total Records': len(analysis['dates']) if analysis['dates'] is not None else 0
        }
        summary_data.append(summary_row)

        # Print summary
        print(f"    Number of files: {analysis['file_count']}")

        if analysis['dates'] is not None:
            print(f"    Start date: {analysis['start_date'].strftime('%Y-%m-%d')}")
            print(f"    End date: {analysis['end_date'].strftime('%Y-%m-%d')}")
            print(f"    Total records: {len(analysis['dates'])}")

            # Plot date distribution
            date_counts = analysis['dates']['date'].dt.date.value_counts().sort_index()

            plt.figure(figsize=(14, 5))
            plt.bar(range(len(date_counts)), date_counts.values)
            plt.xlabel('Date')
            plt.ylabel('Number of Rows')
            plt.title(f'{region} - {deployment.name}: Distribution of Records by Date')
            plt.xticks(range(0, len(date_counts), max(1, len(date_counts)//20)),
                      [str(d) for d in date_counts.index[::max(1, len(date_counts)//20)]],
                      rotation=45, ha='right')
            plt.tight_layout()

            # Save to PDF instead of showing
            pdf.savefig()
            plt.close()

            # Print top dates
            print(f"\n    Top 5 dates by record count:")
            for date, count in date_counts.head().items():
                print(f"      {date}: {count} rows")
        else:
            print(f"    No date data found")

# Close the PDF
pdf.close()



In [ ]:
print(f"\n{'='*80}")
print(f"Analysis complete!")
print(f"{'='*80}")

# Create summary DataFrame
summary_df = pd.DataFrame(summary_data)

# Display the summary table
print(f"\n{'='*80}")
print(f"SUMMARY TABLE")
print(f"{'='*80}\n")
print(summary_df.to_string(index=False))

# Save to CSV
output_file = 'deployment_summary.csv'
summary_df.to_csv(output_file, index=False)
print(f"\n{'='*80}")
print(f"Summary table saved to: {output_file}")
print(f"All plots saved to: {pdf_filename}")
print(f"{'='*80}")
